# Standalone RGB models and training
This walkthrough builds a classifier and shows the full training, resume, evaluation and inference commands. The new models are research extensions, not reproduced manuscript experiments. Install the package first with `pip install -e .`. Run from the repository root or notebooks directory.

In [ ]:
from pathlib import Path
import os
import sys
import subprocess
if Path.cwd().name == "notebooks":
    os.chdir("..")
assert Path("pyproject.toml").exists()
from wbc.models import MODEL_CATALOG, build_model
from wbc.engine import read_config
import torch
torch.set_num_threads(2)
list(MODEL_CATALOG)


## Inspect an actual classifier
The following randomly initialized compact model performs a shape check on synthetic tensors. It is not a trained model and produces no research result.

In [ ]:
config = read_config("configs/rgb_pbc.yaml")
config["model"].update(backbone="compact_cnn", pretrained=False)
config["image_size"] = 64
model = build_model(config).eval()
with torch.no_grad():
    output = model(torch.rand(2, 1, 3, 64, 64))
print("logits:", tuple(output["logits"].shape))
print("tokens:", tuple(output["tokens"].shape))
print("parameters:", sum(p.numel() for p in model.parameters()))


## Train on prepared data
Prepare the dataset and fixed split using the repository dataset guide, then edit roots and manifest paths in the RGB configs. The next cell only prints commands; set `EXECUTE = True` after choosing a fresh output directory and verifying your data. Full training uses the configured epoch count.

In [ ]:
EXECUTE = False
commands = [
    ["-m", "wbc.cli", "train", "--config", "configs/rgb_pbc.yaml", "--out", "runs/notebook_rgb"],
    ["-m", "wbc.cli", "evaluate", "--checkpoint", "runs/notebook_rgb/calibrated.pt", "--manifest", "data/splits/pbc.csv", "--split", "test", "--out", "runs/notebook_retest"],
]
for command in commands:
    print(" ".join([sys.executable, *command]))
    if EXECUTE:
        subprocess.run([sys.executable, *command], check=True)


## Resume, compare, and export
Keep the original config/schedule for resume. Choose ensemble members and weights before viewing test outcomes. Use new output directories for exports.

```bash
wbc train --config configs/rgb_pbc.yaml --out runs/notebook_rgb --resume runs/notebook_rgb/resume.pt
python scripts/train_baselines.py --config configs/pbc.yaml --backbones resnet18 densenet121 swin_t --out runs/model_matrix
wbc summarize --runs runs/model_a runs/model_b --out runs/comparison.csv
wbc ensemble --runs runs/model_a runs/model_b --weights 1 1 --out runs/ensemble
wbc infer --checkpoint runs/notebook_rgb/calibrated.pt --images path/to/cells --features --out runs/features
```

The planner writes configurations by default; add `--execute` to train. Feature exports contain IDs and per-scale representations and load with `numpy.load(..., allow_pickle=False)`. See `docs/models_and_training.md` for assumptions and limits.